# 01 — LMSYS-Chat-1M dataset inspection

Purpose:
- verify access to the primary dataset;
- inspect the **actual runtime schema** without assuming field names;
- stream only a very small sample;
- identify the conversation/message field;
- extract the first user/assistant pair into a local, flattened sample;
- record the action in the experiment log.

This notebook does **not** run a classifier and does **not** produce research results.

## Before running

1. Sign in to Hugging Face.
2. Accept the access/licence conditions on `lmsys/lmsys-chat-1m`.
3. From a terminal in this project environment, run:

```bash
hf auth login
```

Keep the token out of the notebook and source code.

In [ ]:
from pathlib import Path
import sys

# Make `src` importable when this notebook is launched from notebooks/.
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

In [ ]:
from src.config import (
    DATASET_ID,
    DATASET_REVISION,
    DATASET_SPLIT,
    INSPECTION_SAMPLE_SIZE,
    RANDOM_SEED,
    SAMPLES_DIR,
)
from src.data_loading import (
    append_experiment_log,
    build_safe_flat_sample,
    load_lmsys_stream,
    preview_text,
    schema_summary,
    take_examples,
)

print("Dataset:", DATASET_ID)
print("Split:", DATASET_SPLIT)
print("Revision:", DATASET_REVISION)
print("Inspection sample size:", INSPECTION_SAMPLE_SIZE)
print("Random seed reserved for later sampling:", RANDOM_SEED)

## 1. Stream the dataset

Streaming avoids downloading the full dataset for this first inspection.  
At this stage we simply take the first 25 examples because the goal is schema verification, not representative sampling.

In [ ]:
stream = load_lmsys_stream()
stream

In [ ]:
examples = take_examples(stream, INSPECTION_SAMPLE_SIZE)
print(f"Loaded {len(examples)} streamed examples.")

## 2. Inspect top-level fields safely

The next table shows only field names, Python types, and non-null counts.  
It deliberately avoids printing every raw value.

In [ ]:
schema_df = schema_summary(examples)
schema_df

## 3. Detect the conversational field and flatten the first exchange

The extractor supports common schemas such as:
- `role` + `content`;
- `from` + `value`.

It does not assume which one LMSYS is currently exposing.

In [ ]:
safe_df = build_safe_flat_sample(examples)

safe_df[
    ["conversation_field", "model", "language", "redacted", "n_messages"]
].head(10)

In [ ]:
print("Detected conversation field counts:")
print(safe_df["conversation_field"].value_counts(dropna=False))

## 4. Preview only a few truncated exchanges

This is for manual verification that the user and assistant fields have been extracted correctly.
Do not interpret these few examples as dataset statistics or findings.

In [ ]:
for idx, row in safe_df.head(3).iterrows():
    print(f"\nExample {idx + 1}")
    print("USER:     ", preview_text(row["user_text"]))
    print("ASSISTANT:", preview_text(row["assistant_text"]))

## 5. Check extraction completeness in this tiny inspection sample

These counts are **diagnostic only**. They are not research results and should not be reported as representative of LMSYS-Chat-1M.

In [ ]:
diagnostic = {
    "rows": len(safe_df),
    "nonempty_user_text": int(safe_df["user_text"].fillna("").str.len().gt(0).sum()),
    "nonempty_assistant_text": int(safe_df["assistant_text"].fillna("").str.len().gt(0).sum()),
}
diagnostic

## 6. Save the local inspection sample

Only flattened research-relevant fields are written.  
The project `.gitignore` prevents dataset samples from being committed accidentally.

In [ ]:
sample_path = SAMPLES_DIR / "lmsys_inspection_sample.csv"
safe_df.to_csv(sample_path, index=False)
print("Saved:", sample_path)

## 7. Log the implementation step

The log stores configuration/metadata only, not conversation text.

In [ ]:
log_path = append_experiment_log(
    event="dataset_inspection",
    details={
        "dataset_id": DATASET_ID,
        "dataset_revision": DATASET_REVISION,
        "split": DATASET_SPLIT,
        "sample_size_requested": INSPECTION_SAMPLE_SIZE,
        "sample_size_loaded": len(examples),
        "conversation_fields_found": sorted(
            {str(v) for v in safe_df["conversation_field"].dropna().unique().tolist()}
        ),
        "saved_sample": str(sample_path),
        "results_claimed": False,
    },
)

print("Log:", log_path)
print("No classifier experiment has been run.")